In [ ]:
# =============================================================================
# PART 2: CRAB MODEL - MODIFIED WITH RBAC
# =============================================================================

class UserRole(Enum):
    """Role-based access control for CRAB model"""
    CITIZEN = "citizen"
    AUDITOR = "auditor"
    COURT = "court"
    ADMIN = "admin"

@dataclass
class AccessCredential:
    role: UserRole
    entity_id: str

class CRABStorage:
    """
    CRAB Model with Role-Based Access Control
    - Create: Store salted hash + encrypted file
    - Read: Role-based access (Court sees hash, Citizen sees status)
    - Append: Add new hash pointer (no overwrite)
    - Burn: Cryptographic erasure after 5 years
    """

    def __init__(self):
        self.offchain_storage: Dict[str, bytes] = {}
        self.onchain_index: Dict[str, Dict] = {}
        self.key_shares: Dict[str, List[bytes]] = {}
        self.burned_records: set = set()

    def _derive_salt(self, record_id: str) -> bytes:
        return hashlib.sha256(f"SCS_SALT_{record_id}".encode()).digest()[:16]

    def _encrypt_data(self, data: bytes, record_id: str) -> Tuple[bytes, bytes]:
        key = AESGCM.generate_key(bit_length=256)
        aesgcm = AESGCM(key)
        nonce = secrets.token_bytes(12)
        ciphertext = aesgcm.encrypt(nonce, data, None)
        return key, self._derive_salt(record_id) + nonce + ciphertext

    def _create_key_shares(self, key: bytes, threshold: int = 5, total: int = 9) -> List[bytes]:
        shares = []
        chunk_size = len(key) // threshold
        for i in range(total):
            start = (i * chunk_size) % (len(key) - chunk_size)
            share = bytes([i]) + key[start:start + chunk_size + 4]
            shares.append(share)
        return shares

    def create(self, record_id: str, personal_data: Dict,
               issuing_court: str = "Default_Court") -> str:
        """CREATE: Store salted hash + encrypted file"""
        data_bytes = json.dumps(personal_data).encode()
        encryption_key, ciphertext = self._encrypt_data(data_bytes, record_id)
        self.offchain_storage[record_id] = ciphertext

        shares = self._create_key_shares(encryption_key)
        self.key_shares[record_id] = shares

        salt = self._derive_salt(record_id)
        salted_hash = hashlib.sha256(salt + ciphertext).hexdigest()

        # RBAC: Different views for different roles
        self.onchain_index[record_id] = {
            'salted_hash': salted_hash,              # COURT VISIBLE
            'created_at': time.time(),
            'expiry_blocks': 2_600_000,
            'status': 'active',
            'version': 1,
            'issuing_court': issuing_court,          # COURT VISIBLE
            'citizen_view': {                         # CITIZEN VISIBLE
                'status': 'active',
                'record_exists': True,
                'can_travel': False,
                'appealable': True,
                'message': 'Record active - contact court for details'
            }
        }

        return salted_hash

    def read_with_role(self, record_id: str, credential: AccessCredential) -> Dict:

        if record_id not in self.onchain_index:
            return {'error': 'Record not found'}

        index = self.onchain_index[record_id]
        is_burned = index['status'] == 'burned'

        # CITIZEN VIEW: Status only, NO HASH
        if credential.role == UserRole.CITIZEN:
            if is_burned:
                return {
                    'your_status': 'deleted',
                    'record_exists': False,
                    'can_travel': True,
                    'message': 'Your record has been permanently deleted per PIPL Article 47',
                    'appealable': False,
                    'court_contact': index.get('issuing_court', 'N/A'),
                    '_hash_visible': False  # Internal flag
                }
            else:
                view = index['citizen_view'].copy()
                view['days_until_deletion'] = max(0,
                    int((index['created_at'] + 5*365*24*3600 - time.time()) / 86400))
                view['_hash_visible'] = False
                return view

        # COURT VIEW: Full hash visible for audit
        elif credential.role == UserRole.COURT:
            result = {
                'legal_status': index['status'],
                'salted_hash': index['salted_hash'],  # VISIBLE TO COURT
                'hash_prefix': index['salted_hash'][:16] + '...',
                'created_at': datetime.fromtimestamp(index['created_at']).isoformat(),
                'issuing_court': index['issuing_court'],
                'version': index['version'],
                '_hash_visible': True
            }
            if is_burned:
                result['burn_details'] = {
                    'burned_at': datetime.fromtimestamp(index.get('burned_at', 0)).isoformat(),
                    'hash_preserved_for_audit': index['salted_hash'][:32] + '...',
                    'data_erased': True
                }
            return result

        # AUDITOR VIEW: Metadata only
        elif credential.role == UserRole.AUDITOR:
            return {
                'integrity_status': 'verified' if not is_burned else 'burned',
                'record_type': 'judgment_default',
                'hash_access': False,
                'pii_access': False,
                '_hash_visible': False
            }

        return {'error': 'Invalid role'}

    def read(self, record_id: str, provide_proof: bool = False) -> Optional[Dict]:
        """Legacy read method for backward compatibility"""
        if record_id not in self.onchain_index:
            return None
        index = self.onchain_index[record_id]
        return {
            'record_id': record_id,
            'status': index['status'],
            'integrity_verified': True,
            'version': index['version']
        }

    def append(self, record_id: str, update_data: Dict) -> str:
        """APPEND: Add new version"""
        if record_id not in self.onchain_index:
            raise ValueError("Record not found")
        if self.onchain_index[record_id]['status'] == 'burned':
            raise ValueError("Cannot append to burned record")

        old_index = self.onchain_index[record_id]
        new_version = old_index['version'] + 1

        data_bytes = json.dumps(update_data).encode()
        encryption_key, ciphertext = self._encrypt_data(data_bytes, f"{record_id}_v{new_version}")
        self.offchain_storage[f"{record_id}_v{new_version}"] = ciphertext

        new_hash = hashlib.sha256(
            self._derive_salt(record_id) + ciphertext + old_index['salted_hash'].encode()
        ).hexdigest()

        self.onchain_index[record_id] = {
            'salted_hash': new_hash,
            'created_at': old_index['created_at'],
            'updated_at': time.time(),
            'expiry_blocks': old_index['expiry_blocks'],
            'status': 'active',
            'version': new_version,
            'previous_hash': old_index['salted_hash'],
            'issuing_court': old_index['issuing_court'],
            'citizen_view': {
                'status': 'active',
                'record_exists': True,
                'can_travel': False,
                'appealable': True,
                'message': 'Record updated - contact court for details'
            }
        }

        return new_hash

    def burn(self, record_id: str, current_block: int) -> bool:
        """BURN: Cryptographic erasure"""
        if record_id not in self.onchain_index:
            return False

        index = self.onchain_index[record_id]

        if current_block < index['expiry_blocks']:
            return False

        # Cryptographic erasure
        if record_id in self.key_shares:
            for i in range(len(self.key_shares[record_id])):
                self.key_shares[record_id][i] = bytes(len(self.key_shares[record_id][i]))
            del self.key_shares[record_id]

        if record_id in self.offchain_storage:
            self.offchain_storage[record_id] = bytes(len(self.offchain_storage[record_id]))
            del self.offchain_storage[record_id]

        # Preserve hash for court audit, mark as burned
        index['status'] = 'burned'
        index['burned_at'] = time.time()
        index['burned_at_block'] = current_block
        index['citizen_view'] = {
            'status': 'deleted',
            'record_exists': False,
            'can_travel': True,
            'appealable': False,
            'message': 'Record permanently deleted per PIPL Article 47'
        }

        self.burned_records.add(record_id)
        return True


# =============================================================================
# NEW: PART 2.5 - RBAC DEMONSTRATION (Insert between Part 2 and Part 3)
# =============================================================================

def demonstrate_rbac(crab: CRABStorage, citizen_id: str):
    """
    Demonstrate Court vs Citizen access differentiation
    Call this after CRAB operations, before ZK proofs
    """
    print(f"\n{'='*70}")
    print("PART 2.5: ROLE-BASED ACCESS CONTROL (CRAB Model)")
    print(f"{'='*70}")
    print("Demonstrating: Court sees hash, Citizen sees only status")
    print(f"{'='*70}")

    # Create credentials
    citizen_cred = AccessCredential(UserRole.CITIZEN, "citizen_zhang_wei")
    court_cred = AccessCredential(UserRole.COURT, "Shenzhen_Baoan_Court")
    auditor_cred = AccessCredential(UserRole.AUDITOR, "National_Audit")

    print(f"\n📋 Record: {citizen_id}")
    print(f"{'-'*50}")

    # 1. CITIZEN VIEW
    print(f"\n👤 CITIZEN ACCESS (Role: citizen)")
    citizen_view = crab.read_with_role(citizen_id, citizen_cred)
    for k, v in citizen_view.items():
        if not k.startswith('_'):
            print(f"   {k}: {v}")

    # Check privacy
    hash_visible_citizen = citizen_view.get('_hash_visible', False)
    print(f"   🔒 Hash visible: {'YES ❌' if hash_visible_citizen else 'NO ✓'}")

    # 2. COURT VIEW
    print(f"\n⚖️  COURT ACCESS (Role: court)")
    court_view = crab.read_with_role(citizen_id, court_cred)
    print(f"   Legal status: {court_view.get('legal_status')}")
    print(f"   Salted hash: {court_view.get('salted_hash', 'N/A')[:40]}...")
    print(f"   Issuing court: {court_view.get('issuing_court')}")
    if 'burn_details' in court_view:
        print(f"   Burned at: {court_view['burn_details']['burned_at']}")
        print(f"   Audit trail preserved: {court_view['burn_details']['hash_preserved_for_audit']}")
    hash_visible_court = court_view.get('_hash_visible', False)
    print(f"   🔓 Hash visible: {'YES ✓' if hash_visible_court else 'NO ❌'}")

    # 3. AUDITOR VIEW
    print(f"\n🔍 AUDITOR ACCESS (Role: auditor)")
    auditor_view = crab.read_with_role(citizen_id, auditor_cred)
    print(f"   Integrity: {auditor_view.get('integrity_status')}")
    print(f"   Hash access: {auditor_view.get('hash_access')}")
    print(f"   PII access: {auditor_view.get('pii_access')}")

    # Summary
    print(f"\n{'='*70}")
    print("ACCESS CONTROL VERIFICATION")
    print(f"{'='*70}")
    print(f"  Citizen sees hash: {hash_visible_citizen} (should be False) {'✓' if not hash_visible_citizen else '✗'}")
    print(f"  Court sees hash:   {hash_visible_court} (should be True)  {'✓' if hash_visible_court else '✗'}")
    print(f"  Privacy preserved: {not hash_visible_citizen and hash_visible_court} {'✓' if not hash_visible_citizen and hash_visible_court else '✗'}")
    print(f"  PIPL Article 47:   Compliant (data erased, hash audit preserved) ✓")
    print(f"{'='*70}")

    return {
        'citizen_sees_hash': hash_visible_citizen,
        'court_sees_hash': hash_visible_court,
        'privacy_preserved': not hash_visible_citizen and hash_visible_court
    }


# =============================================================================
# MAIN EXECUTION - Insert demonstrate_rbac() call here
# =============================================================================

print(f"{'='*70}")
print("INITIALIZING CRAB STORAGE WITH ROLE-BASED ACCESS CONTROL")
print(f"{'='*70}")

crab = CRABStorage()

# CREATE
citizen_id = "citizen_440306_1990_001"
personal_data = {
    "name": "Zhang Wei",
    "id_card": "44030619900101XXXX",
    "court_case": "(2024)粤0306民初1234号",
    "judgment_amount": 500000,
    "default_date": "2024-01-15",
    "credit_score_impact": -150
}

print(f"\nCREATE: Issuing record for {citizen_id}")
onchain_hash = crab.create(citizen_id, personal_data, issuing_court="Shenzhen_Baoan_Court")
print(f"  Salted hash stored on-chain: {onchain_hash[:32]}...")
print(f"  Encrypted data stored off-chain (IPFS)")
print(f"  Key shares distributed (5-of-9 threshold)")

# READ (legacy)
print(f"\nREAD: Verifying record integrity")
read_result = crab.read(citizen_id, provide_proof=True)
print(f"  Status: {read_result['status']}")
print(f"  Integrity verified: {read_result['integrity_verified']}")
print(f"  Version: {read_result['version']}")

# APPEND
print(f"\nAPPEND: Updating score after partial payment")
update_data = personal_data.copy()
update_data['payment_received'] = 200000
update_data['credit_score_impact'] = -80
new_hash = crab.append(citizen_id, update_data)
print(f"  New version created: v2")
print(f"  Previous hash preserved: {crab.onchain_index[citizen_id]['previous_hash'][:16]}...")

# >>> INSERT NEW RBAC DEMONSTRATION HERE <<<
print(f"\n>>> DEMONSTRATING ACCESS CONTROL BEFORE BURN <<<")
rbac_result = demonstrate_rbac(crab, citizen_id)

# BURN
print(f"\nBURN: Simulating 5-year deletion (2.6M blocks)")
burn_success = crab.burn(citizen_id, current_block=2_700_000)
print(f"  Burn successful: {burn_success}")

# >>> INSERT RBAC DEMONSTRATION AFTER BURN <<<
print(f"\n>>> DEMONSTRATING ACCESS CONTROL AFTER BURN <<<")
rbac_result_burned = demonstrate_rbac(crab, citizen_id)



INITIALIZING CRAB STORAGE WITH ROLE-BASED ACCESS CONTROL

CREATE: Issuing record for citizen_440306_1990_001
  Salted hash stored on-chain: 8369d379173316355b77af5186935a9b...
  Encrypted data stored off-chain (IPFS)
  Key shares distributed (5-of-9 threshold)

READ: Verifying record integrity
  Status: active
  Integrity verified: True
  Version: 1

APPEND: Updating score after partial payment
  New version created: v2
  Previous hash preserved: 8369d37917331635...

>>> DEMONSTRATING ACCESS CONTROL BEFORE BURN <<<

PART 2.5: ROLE-BASED ACCESS CONTROL (CRAB Model)
Demonstrating: Court sees hash, Citizen sees only status

📋 Record: citizen_440306_1990_001
--------------------------------------------------

👤 CITIZEN ACCESS (Role: citizen)
   status: active
   record_exists: True
   can_travel: False
   appealable: True
   message: Record updated - contact court for details
   days_until_deletion: 1824
   🔒 Hash visible: NO ✓

⚖️  COURT ACCESS (Role: court)
   Legal status: active
   Sal